In [1]:
from pyspark.sql import SparkSession
 
# Crear una sesión de Spark
spark = SparkSession.builder \
    .appName("Comprobar Parquets") \
    .getOrCreate()

In [2]:
from notebookutils import mssparkutils

# CONFIGURACIÓN

RUTA_BASE = f"abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/raw/"

archivos_vacios = []
archivos_validos = 0

print(f"Iniciando escaneo profundo en: {RUTA_BASE}...")

# FUNCIÓN RECURSIVA DE BÚSQUEDA
def rastrear_carpeta(ruta_actual):
    global archivos_validos
    
    try:
        elementos = mssparkutils.fs.ls(ruta_actual)
        
        for elemento in elementos:
            if elemento.isDir:
                # Si es una carpeta, entramos dentro de ella (recursividad)
                rastrear_carpeta(elemento.path)
            else:
                # Si es un archivo parquet, comprobamos su tamaño
                if elemento.name.endswith(".parquet"):
                    if elemento.size == 0:
                        archivos_vacios.append(elemento.path)
                    else:
                        archivos_validos += 1
    except Exception as e:
        print(f"Error al leer la ruta {ruta_actual}: {e}")

# EJECUCIÓN Y RESULTADOS
rastrear_carpeta(RUTA_BASE)

print("\n" + "="*40)
print("INFORME DE CALIDAD DEL DATA LAKE")
print("="*40)
print(f"Archivos Parquet sanos (con datos): {archivos_validos}")

if len(archivos_vacios) > 0:
    print(f"¡Alerta! Se encontraron {len(archivos_vacios)} archivos vacíos (0 bytes):")
    for archivo in archivos_vacios:
        # Imprimimos solo la parte final de la ruta para que sea fácil de leer
        ruta_corta = archivo.split("/raw/")[1]
        print(f"   -> {ruta_corta}")
else:
    print("¡Todo perfecto! No se encontró ningún archivo de 0 bytes.")

In [4]:
import json
from notebookutils import mssparkutils

# CONFIGURACIÓN

# Necesitamos leer el JSON que usamos en la pipeline
RUTA_JSON = f"abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/lista_urls.json"
RUTA_BASE = f"abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/raw/"

print("Analizando Lista Maestra vs Realidad del Data Lake...")

# LEER LA LISTA MAESTRA
texto_json = mssparkutils.fs.head(RUTA_JSON, 1024000) # Leemos el JSON
lista_esperada = json.loads(texto_json)

archivos_faltantes = []
archivos_sanos = 0

# RECONCILIACIÓN (CRUCE)
for item in lista_esperada:
    anio = item['anio']
    mes = item['mes']
    tipo = item['tipo']
    archivo = item['url_part']
    
    # Construimos la ruta donde DEBERÍA estar el archivo
    ruta_destino = f"{RUTA_BASE}year={anio}/month={mes}/type={tipo}/{archivo}"
    
    try:
        # Preguntamos si el archivo existe y cuánto pesa
        info = mssparkutils.fs.ls(ruta_destino)[0]
        if info.size > 0:
            archivos_sanos += 1
        else:
            archivos_faltantes.append(f"{archivo} (Existe pero corrupto/0 bytes)")
    except:
        # Si da error, es que la pipeline falló (Error 403) y no lo creó
        archivos_faltantes.append(f"{archivo} (Falta en el Data Lake)")

# INFORME FINAL
print("\n" + "="*45)
print("INFORME DE RECONCILIACIÓN")
print("="*45)
print(f"Total en JSON (Esperados): {len(lista_esperada)}")
print(f"Descargados con éxito: {archivos_sanos}")
print(f"Fallidos (Posible 403): {len(archivos_faltantes)}")

if archivos_faltantes:
    print("\nLISTA EXACTA DE LOS 20 ARCHIVOS A RECUPERAR:")
    for f in archivos_faltantes:
        print(f"  - {f}")

In [3]:
spark.stop()